# Atividade 2.1 - Zeros de Funções Reais (Questão A)

**Objetivo:** Comparação de métodos numéricos (Bissecção, Posição Falsa, Ponto Fixo, Newton e Secante) para encontrar raízes de funções reais.

Reprodução das tabelas dos **Exemplos 18, 19, 20, 21 e 22** conforme solicitado, seguindo os critérios de parada $\epsilon_1, \epsilon_2$ e número máximo de iterações.

In [32]:
import math
import time
import pandas as pd
import numpy as np

#configuração para as tabelas ficarem mais bonitas e legíveis
pd.set_option('display.float_format', '{:.8f}'.format)
pd.set_option('display.colheader_justify', 'center')

#funcoees dos Métodos Numéricos

def metodo_bisseccao(f, a, b, eps, itmax=100):
    start = time.perf_counter()
    if f(a) * f(b) > 0: return {'Status': 'Falha (Sinais iguais)'}
    
    delta_x = abs(b - a)
    k = 0
    x = a
    fx = f(x)
    
    while k < itmax:
        delta_x = delta_x / 2
        x = a + delta_x
        fx = f(x)
        
        if (delta_x <= eps) or (abs(fx) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x, 'Erro f(x)': abs(fx), 'Tempo (s)': end-start}
            
        if f(a) * fx < 0:
            b = x
        else:
            a = x
        k += 1
    
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(fx), 'Tempo (s)': end-start}

def metodo_posicao_falsa(f, a, b, eps, itmax=100):
    start = time.perf_counter()
    if f(a) * f(b) > 0: return {'Status': 'Falha (Sinais iguais)'}
    
    k = 0
    while k < itmax:
        if abs(b - a) <= eps: break
        
        fa, fb = f(a), f(b)
        if (fb - fa) == 0: return {'Status': 'Erro: Divisão por zero'}
        
        x = (a * fb - b * fa) / (fb - fa)
        fx = f(x)
        
        if abs(fx) <= eps:
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x, 'Erro f(x)': abs(fx), 'Tempo (s)': end-start}
            
        if fa * fx < 0:
            b = x
        else:
            a = x
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': end-start}

def metodo_mpf(f, phi, x0, eps, itmax=100):
    start = time.perf_counter()
    x = x0
    k = 0
    
    if abs(f(x)) <= eps:
        return {'k (Iter)': 0, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': 0}
        
    while k < itmax:
        try:
            x_new = phi(x)
        except:
             return {'Status': 'Erro de Domínio'}

        if (abs(x_new - x) <= eps) or (abs(f(x_new)) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x_new, 'Erro f(x)': abs(f(x_new)), 'Tempo (s)': end-start}
            
        x = x_new
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': end-start}

def metodo_newton(f, df, x0, eps, itmax=100):
    start = time.perf_counter()
    x = x0
    k = 0
    
    if abs(f(x)) <= eps:
        return {'k (Iter)': 0, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': 0}
        
    while k < itmax:
        deriv = df(x)
        if deriv == 0: return {'Status': 'Erro: Derivada zero'}
        
        x_new = x - (f(x) / deriv)
        
        if (abs(x_new - x) <= eps) or (abs(f(x_new)) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x_new, 'Erro f(x)': abs(f(x_new)), 'Tempo (s)': end-start}
            
        x = x_new
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': end-start}

def metodo_secante(f, x0, x1, eps, itmax=100):
    start = time.perf_counter()
    fx0, fx1 = f(x0), f(x1)
    
    if abs(fx0) <= eps: return {'k (Iter)': 0, 'Raiz Aproximada': x0, 'Erro f(x)': abs(fx0), 'Tempo (s)': 0}
    if abs(fx1) <= eps: return {'k (Iter)': 0, 'Raiz Aproximada': x1, 'Erro f(x)': abs(fx1), 'Tempo (s)': 0}
    
    k = 0
    while k < itmax:
        den = fx1 - fx0
        if den == 0: return {'Status': 'Erro: Divisão por zero'}
        
        x_new = x1 - (fx1 * (x1 - x0) / den)
        fx_new = f(x_new)
        
        if (abs(x_new - x1) <= eps) or (abs(fx_new) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x_new, 'Erro f(x)': abs(fx_new), 'Tempo (s)': end-start}
            
        x0, fx0 = x1, fx1
        x1, fx1 = x_new, fx_new
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x1, 'Erro f(x)': abs(fx1), 'Tempo (s)': end-start}

def executar_bonito(titulo, f, df, phi, a, b, x0, x1_sec, eps):
    resultados = []
    
    #executa cada metodo e coleta resultados
    #bisseccão
    r = metodo_bisseccao(f, a, b, eps)
    if r: resultados.append(['Bissecção'] + list(r.values())[:4]) # Pega apenas os 4 primeiros valores
    
    #posicao falsa
    r = metodo_posicao_falsa(f, a, b, eps)
    if r: resultados.append(['Posição Falsa'] + list(r.values())[:4])
    
    #MPF
    r = metodo_mpf(f, phi, x0, eps)
    if r: resultados.append(['MPF'] + list(r.values())[:4])
    
    #Newton
    r = metodo_newton(f, df, x0, eps)
    if r: resultados.append(['Newton'] + list(r.values())[:4])
    
    #secante
    r = metodo_secante(f, x0, x1_sec, eps)
    if r: resultados.append(['Secante'] + list(r.values())[:4])
    
    #cria tabela formatada
    df_res = pd.DataFrame(resultados, columns=['Método', 'Iterações (k)', 'Raiz Encontrada', 'Erro Abs |f(x)|', 'Tempo (s)'])
    
    #exibe a tabela
    print("\n" + "="*60)
    print(f"{titulo.center(60)}")
    print(f"Intervalo: [{a}, {b}] | Precisão: {eps}")
    print("="*60)
    display(df_res.T)

In [33]:
#EXEMPLO 18
#f(x) = e^(-x^2) - cos(x)
#Intervalo: [1, 2]; Eps = 10^-4
#Phi(x) = arccos(e^(-x^2))

def f18(x): 
    return math.exp(-x**2) - math.cos(x)

def df18(x): 
    #derivada: -2x*e^(-x^2) + sen(x)
    return -2*x*math.exp(-x**2) + math.sin(x)

def phi18(x): 
    #MPF: x = arccos(e^(-x^2))
    try:
        val = math.exp(-x**2)
        #protecao de domínio para arccos [-1, 1]
        if val > 1: val = 1
        if val < -1: val = -1
        return math.acos(val)
    except: return x

#parametros do exemplo
a, b = 1.0, 2.0
eps = 1e-4
x0 = 1.5      #ponto medio
x1_sec = 2.0  #extremidade para secante

executar_bonito("EXEMPLO 18", f18, df18, phi18, a, b, x0, x1_sec, eps)


                         EXEMPLO 18                         
Intervalo: [1.0, 2.0] | Precisão: 0.0001


,0,1,2,3,4
Método,Bissecção,Posição Falsa,MPF,Newton,Secante
Iterações (k),9,6,6,2,3
Raiz Encontrada,1.44726562,1.44735707,1.44751711,1.44741635,1.44742556
Erro Abs |f(x)|,0.00009455,0.00003639,0.00006542,0.00000132,0.00000718
Tempo (s),0.00004192,0.00002199,0.00001974,0.00001299,0.00000998


In [34]:
# EXEMPLO 19
# f(x) = x^3 - x - 1
def f19(x): return x**3 - x - 1
def df19(x): return 3*x**2 - 1
def phi19(x): return math.pow(x + 1, 1/3) # x = (x+1)^(1/3)

#parametros: intervalo [1, 2], Eps = 10^-6
#chute inicial (x0): ponto médio (1.5)
executar_bonito("EXEMPLO 19", f19, df19, phi19, 1.0, 2.0, 1.5, 2.0, 1e-6)


                         EXEMPLO 19                         
Intervalo: [1.0, 2.0] | Precisão: 1e-06


,0,1,2,3,4
Método,Bissecção,Posição Falsa,MPF,Newton,Secante
Iterações (k),20,17,9,3,5
Raiz Encontrada,1.32471752,1.32471776,1.32471801,1.32471817,1.32471803
Erro Abs |f(x)|,0.00000186,0.00000083,0.00000023,0.00000092,0.00000031
Tempo (s),0.00003401,0.00002496,0.00001484,0.00000838,0.00000820


In [35]:
#EXEMPLO 20
#f(x) = 4sen(x) - e^x
def f20(x): return 4 * math.sin(x) - math.exp(x)
def df20(x): return 4 * math.cos(x) - math.exp(x)
def phi20(x): 
    #isolando x: 4sen(x) = e^x => sen(x) = e^x / 4 => x = arcsen(e^x / 4)
    try:
        val = math.exp(x) / 4
        return math.asin(val)
    except: return x 

#parametros: intervalo [0, 1], eps = 10^-5
executar_bonito("EXEMPLO 20", f20, df20, phi20, 0.0, 1.0, 0.5, 1.0, 1e-5)


                         EXEMPLO 20                         
Intervalo: [0.0, 1.0] | Precisão: 1e-05


,0,1,2,3,4
Método,Bissecção,Posição Falsa,MPF,Newton,Secante
Iterações (k),16,8,11,3,6
Raiz Encontrada,0.37055969,0.37055883,0.37056258,0.37055808,0.37055823
Erro Abs |f(x)|,0.00000364,0.00000167,0.00001022,0.00000003,0.00000030
Tempo (s),0.00003921,0.00001498,0.00001560,0.00000892,0.00000828


In [36]:
#EXEMPLO 21
#f(x) = x * log10(x) - 1
def f21(x): return x * math.log10(x) - 1
def df21(x): return math.log10(x) + (1 / math.log(10)) # Regra do produto
def phi21(x): 
    #isolando x: log10(x) = 1/x => x = 10^(1/x)
    try: return math.pow(10, 1/x)
    except: return x

#parametros: intervalo [2, 3], Eps = 10^-7
executar_bonito("EXEMPLO 21", f21, df21, phi21, 2.0, 3.0, 2.5, 3.0, 1e-7)


                         EXEMPLO 21                         
Intervalo: [2.0, 3.0] | Precisão: 1e-07


,0,1,2,3,4
Método,Bissecção,Posição Falsa,MPF,Newton,Secante
Iterações (k),21,5,100,2,3
Raiz Encontrada,2.50618410,2.50618403,2.50618285,2.50618415,2.50618415
Erro Abs |f(x)|,0.00000004,0.00000010,0.00000108,0.00000000,0.00000000
Tempo (s),0.00003296,0.00000916,0.00005846,0.00000759,0.00000451


In [37]:
#EXEMPLO 22
#f(x) = (x - 1)^2 * (x - 1.5)
#isso expande para: x^3 - 3.5x^2 + 4x - 1.5
#usei a forma fatorada para que eu pudesse garantir uma certa precisão matemática (evitando o erro de digitação 3.2 vs 3.5)

def f22(x): 
    return (x - 1)**2 * (x - 1.5)

def df22(x): 
    #derivada pela regra do produto ou do polinômio expandido:
    #3x^2 - 7x + 4
    return 3*(x**2) - 7*x + 4

def phi22(x):
    #MPF para a raiz 1.5
    #isolando x da forma expandida x^3 - 3.5x^2 + 4x - 1.5 = 0
    #4x = -x^3 + 3.5x^2 + 1.5 => x = (3.5x^2 - x^3 + 1.5) / 4
    return (3.5*(x**2) - x**3 + 1.5) / 4

#paarametros: intervalo [1.1, 2.0] para focar na raiz 1.5 (onde há troca de sinal)
#se usar [0, 2], a raiz dupla em 1.0 atrapalha a bissecção.
executar_bonito("EXEMPLO 22", f22, df22, phi22, 1.1, 2.0, 1.6, 2.0, 1e-6)


                         EXEMPLO 22                         
Intervalo: [1.1, 2.0] | Precisão: 1e-06


,0,1,2,3,4
Método,Bissecção,Posição Falsa,MPF,Newton,Secante
Iterações (k),16,55,100,4,6
Raiz Encontrada,1.49999847,1.49999679,1.50010848,1.50000000,1.50000042
Erro Abs |f(x)|,0.00000038,0.00000080,0.00002713,0.00000000,0.00000010
Tempo (s),0.00005766,0.00009232,0.00010615,0.00001257,0.00000998


# Atividade 2.1 - Questão B: Polinômios (Exemplo 1)

**Objetivo:** Adaptar o pseudocódigo do Método de Newton para Polinômios (que utiliza o esquema de Horner/Briot-Ruffini para avaliação eficiente) para os métodos da **Secante** e **Ponto Fixo (MPF)**.

$$P_5(x) = x^5 - 3.7x^4 + 7.4x^3 - 10.8x^2 + 10.8x - 6.8 = 0$$

* **Intervalo:** $(1, 2)$
* **Chute inicial ($x_0$):** $1.5$
* **Precisão ($\epsilon$):** $10^{-6}$

### Explicação da Adaptação:

1.  **Algoritmo Original (Newton para Polinômios):** Calcula simultaneamente $P(x)$ (variável `b`) e $P'(x)$ (variável `c`) iterando pelos coeficientes. Isso evita o cálculo custoso de potências ($x^n$).
2.  **Adaptação para Secante:** O método da Secante não precisa da derivada $P'(x)$. Adaptamos o código removendo o cálculo da variável `c`. Usaremos o esquema de Horner apenas para calcular $P(x)$ nos pontos $x_k$ e $x_{k-1}$.
3.  **Adaptação para MPF:** O MPF requer uma função de iteração $\varphi(x)$. Usaremos o esquema de Horner para avaliar o polinômio dentro da função de iteração escolhida. Para este exemplo, isolaremos o termo linear para criar a função de iteração: $\varphi(x) = \frac{-x^5 + 3.7x^4 - 7.4x^3 + 10.8x^2 + 6.8}{10.8}$.

In [38]:
import time
import pandas as pd
import math

#coeficientes do polinomio P5(x) do exemplo 1
#P(x) = 1x^5 - 3.7x^4 + 7.4x^3 - 10.8x^2 + 10.8x - 6.8
#ordem: [a5, a4, a3, a2, a1, a0]
coeficientes = [1.0, -3.7, 7.4, -10.8, 10.8, -6.8]

def horner_newton(coeffs, x):
    """
    avalia P(x) e P'(x) simultaneamente usando o algoritmo de briot-ruffini (horner).
    baseado no pseudocódigo do livro para o metodo de newton em polinômios.
    retorna: (P(x), P'(x))
    """
    n = len(coeffs) - 1
    b = coeffs[0]
    c = b
    
    #loop para calcular b (coeficientes do quociente/valor da funcao) 
    #e c (valor da derivada) simultaneamente.
    #o loop vai do segundo coeficiente até o penúltimo para c, e até o fim para b
    for i in range(1, n):
        b = coeffs[i] + b * x
        c = b + c * x
        
    # Último passo para b (termo independente final de P(x))
    b = coeffs[n] + b * x
    
    return b, c

def horner_simples(coeffs, x):
    """
    ADAPTAÇÃO: avalia apenas P(x) usando o esquema de horner.
    removi o calculo da variável 'c' (derivada) que existia no algoritmo de newton
    usado para: secante e MPF.
    """
    b = coeffs[0]
    for i in range(1, len(coeffs)):
        b = coeffs[i] + b * x
    return b

In [39]:
#metodo de newton para polinomios (original)
def newton_poly(coeffs, x0, eps, itmax=100):
    start = time.perf_counter()
    x = x0
    k = 0
    
    while k < itmax:
        #usa a função horner_newton que retorna P(x) e P'(x)
        Px, DPx = horner_newton(coeffs, x)
        
        if abs(Px) <= eps:
            end = time.perf_counter()
            return {'Iter': k, 'Raiz': x, 'Erro': abs(Px), 'Tempo (s)': end-start}
            
        if DPx == 0: return {'Status': 'Erro: Derivada zero'}
        
        delta_x = Px / DPx
        x_new = x - delta_x
        
        if abs(x_new - x) <= eps:
            end = time.perf_counter()
            return {'Iter': k+1, 'Raiz': x_new, 'Erro': abs(horner_simples(coeffs, x_new)), 'Tempo (s)': end-start}
            
        x = x_new
        k += 1
        
    end = time.perf_counter()
    return {'Iter': k, 'Raiz': x, 'Erro': abs(Px), 'Tempo (s)': end-start}

#2: metodo da secante adaptado (usa o  horner simples) ---
def secante_poly(coeffs, x0, x1, eps, itmax=100):
    start = time.perf_counter()
    
    #ADAPTAÇÃO: usa horner_simples em no lugar de calcular potências
    fx0 = horner_simples(coeffs, x0)
    fx1 = horner_simples(coeffs, x1)
    
    if abs(fx0) <= eps: return {'Iter': 0, 'Raiz': x0, 'Erro': abs(fx0), 'Tempo (s)': 0}
    if abs(fx1) <= eps: return {'Iter': 0, 'Raiz': x1, 'Erro': abs(fx1), 'Tempo (s)': 0}
    
    k = 0
    while k < itmax:
        den = fx1 - fx0
        if den == 0: return {'Status': 'Divisão por zero'}
        
        x_new = x1 - (fx1 * (x1 - x0) / den)
        fx_new = horner_simples(coeffs, x_new) 
        
        if abs(x_new - x1) <= eps or abs(fx_new) <= eps:
            end = time.perf_counter()
            return {'Iter': k+1, 'Raiz': x_new, 'Erro': abs(fx_new), 'Tempo (s)': end-start}
            
        x0, fx0 = x1, fx1
        x1, fx1 = x_new, fx_new
        k += 1
        
    end = time.perf_counter()
    return {'Iter': k, 'Raiz': x1, 'Erro': abs(fx1), 'Tempo (s)': end-start}

#3: MPF adaptado (phi usa horner)
def mpf_poly(coeffs, x0, eps, itmax=100):
    """
    funcao de iteração phi(x) derivada de P(x) = 0 isolando o termo 10.8x:
    10.8x = -x^5 + 3.7x^4 - 7.4x^3 + 10.8x^2 + 6.8
    x = (termos restantes) / 10.8
    """
    start = time.perf_counter()
    x = x0
    k = 0
    
    #coeficientes para calcular o numerador de phi usando Horner:
    #numerador: -1x^5 + 3.7x^4 - 7.4x^3 + 10.8x^2 + 0x + 6.8
    #note que o termo x^1 linear no numerador é 0, pois foi isolado à esquerda.
    coefs_phi = [-1.0, 3.7, -7.4, 10.8, 0.0, 6.8]
    divisor = 10.8
    
    if abs(horner_simples(coeffs, x)) <= eps:
         return {'Iter': 0, 'Raiz': x, 'Erro': abs(horner_simples(coeffs, x)), 'Tempo (s)': 0}

    while k < itmax:
        # ADAPTAÇÃO: avalia o numerador eficientemente
        numerador = horner_simples(coefs_phi, x)
        x_new = numerador / divisor
        
        erro = abs(horner_simples(coeffs, x_new))
        
        if abs(x_new - x) <= eps or erro <= eps:
            end = time.perf_counter()
            return {'Iter': k+1, 'Raiz': x_new, 'Erro': erro, 'Tempo (s)': end-start}
            
        x = x_new
        k += 1
        
    end = time.perf_counter()
    return {'Iter': k, 'Raiz': x, 'Erro': abs(horner_simples(coeffs, x)), 'Tempo (s)': end-start}

In [40]:
#execucao do exemplo 1
#intervalo (1, 2); x0 = 1.5; Precisão 10^-6

x0 = 1.5
eps = 1e-6

#para a secante, precisa de dois pontos. O intervalo é (1, 2)
#usei x0=1.0 e x1=2.0 (os extremos) ou x0=1.5 e x1=1.6 (próximos)
x_sec0 = 1.5
x_sec1 = 1.6

resultados_B = []

#1: newton polinomial
res_newton = newton_poly(coeficientes, x0, eps)
resultados_B.append(['Newton Poly', res_newton['Iter'], res_newton['Raiz'], res_newton['Erro'], res_newton['Tempo (s)']])

#2: secante polinomial
res_sec = secante_poly(coeficientes, x_sec0, x_sec1, eps)
resultados_B.append(['Secante Poly', res_sec['Iter'], res_sec['Raiz'], res_sec['Erro'], res_sec['Tempo (s)']])

#3: MPF polinomial
res_mpf = mpf_poly(coeficientes, x0, eps)
resultados_B.append(['MPF Poly', res_mpf['Iter'], res_mpf['Raiz'], res_mpf['Erro'], res_mpf['Tempo (s)']])

# Tabela
df_B = pd.DataFrame(resultados_B, columns=['Método', 'Iterações', 'Raiz', 'Erro Final', 'Tempo (s)'])

print("--- Comparação Questão B: Polinômio Exemplo 1 ---")
print(f"Polinômio: x^5 - 3.7x^4 + 7.4x^3 - 10.8x^2 + 10.8x - 6.8")
print(f"Raiz esperada aprox: 1.7")
display(df_B)

--- Comparação Questão B: Polinômio Exemplo 1 ---
Polinômio: x^5 - 3.7x^4 + 7.4x^3 - 10.8x^2 + 10.8x - 6.8
Raiz esperada aprox: 1.7


,Método,Iterações,Raiz,Erro Final,Tempo (s)
0,Newton Poly,4,1.70000006,0.00000042,0.00005443
1,Secante Poly,5,1.70000000,0.00000003,0.00003413
2,MPF Poly,13,1.69999977,0.00000169,0.00004484


### Análise Comparativa (Questão B)

**Adaptação do Código:**
A adaptação principal foi substituir o cálculo direto de potências (ex: `x**5`) pelo algoritmo de **Horner (Briot-Ruffini)**.
* No **Newton**, calculamos `b` (função) e `c` (derivada) num único loop.
* Na **Secante** e **MPF**, simplificamos o algoritmo para calcular apenas `b`, economizando operações, pois estes métodos não necessitam da derivada.

**Comparação de Desempenho:**
1.  **Newton:** Tende a ser o mais eficiente em número de iterações (convergência quadrática) e muito rápido por iteração devido ao cálculo otimizado da derivada junto com a função.
2.  **Secante:** Requer mais iterações que o Newton, mas cada iteração é ligeiramente mais barata (calcula apenas $P(x)$, não $P'(x)$). Contudo, no algoritmo otimizado de polinômios, a diferença de custo computacional é pequena, fazendo o Newton geralmente valer mais a pena.
3.  **MPF:** A convergência depende fortemente da função $\varphi(x)$ escolhida. Isolando o termo linear, a convergência costuma ser linear e mais lenta que os métodos baseados em derivada/inclinação.

**Conclusão:** Para polinômios de grau elevado, o método de Newton com o esquema de Horner é a abordagem padrão mais robusta.

### Questão B: Comparação com Polinômio Cúbico (Exemplo 2)

**Polinômio:** $P_3(x) = x^3 - 3x + 3 = 0$

**Casos de Teste (conforme página 15 do PDF):**
1.  **Caso (i):** $x_0 = -0.8$ (Próximo de um ponto crítico, convergência difícil/lenta).
2.  **Caso (ii):** $x_0 = -2.0$ (Próximo da raiz, convergência rápida).

**Adaptação para MPF:**
Isolo o termo linear $3x$:
$$3x = x^3 + 3 \implies x = \frac{x^3 + 3}{3}$$
A função de iteração será $\varphi(x) = \frac{x^3 + 3}{3}$. Utilizo o esquema de Horner no numerador ($x^3 + 0x^2 + 0x + 3$).

In [41]:
# P3(x) = 1x^3 + 0x^2 - 3x + 3
coefs_p3 = [1.0, 0.0, -3.0, 3.0]

#precisao
eps_p3 = 1e-6

#funcao auxiliar para MPF do exemplo 2
def mpf_poly_p3(coeffs, x0, eps, itmax=100):
    """
    adaptacao especifica para P3(x):
    phi(x) = (x^3 + 3) / 3
    numerador via horner: coefs [1, 0, 0, 3]
    divisor: 3
    """
    start = time.perf_counter()
    x = x0
    k = 0
    
    #coeficientes do numerador (x^3 + 0x^2 + 0x + 3)
    coefs_phi = [1.0, 0.0, 0.0, 3.0]
    divisor = 3.0
    
    if abs(horner_simples(coeffs, x)) <= eps:
         return {'Iter': 0, 'Raiz': x, 'Erro': abs(horner_simples(coeffs, x)), 'Tempo (s)': 0}

    while k < itmax:
        #x_new = (x^3 + 3) / 3
        numerador = horner_simples(coefs_phi, x)
        x_new = numerador / divisor
        
        erro = abs(horner_simples(coeffs, x_new))
        
        #criterio de parada e protecao contra divergencia explosiva
        if abs(x_new - x) <= eps or erro <= eps:
            end = time.perf_counter()
            return {'Iter': k+1, 'Raiz': x_new, 'Erro': erro, 'Tempo (s)': end-start}
        
        if abs(x_new) > 1e10: #protecao
             end = time.perf_counter()
             return {'Iter': k+1, 'Raiz': x_new, 'Erro': erro, 'Tempo (s)': end-start, 'Status': 'Divergiu'}

        x = x_new
        k += 1
        
    end = time.perf_counter()
    return {'Iter': k, 'Raiz': x, 'Erro': abs(horner_simples(coeffs, x)), 'Tempo (s)': end-start}


#CASO (i): x0 = -0.8
x0_i = -0.8
x1_sec_i = -0.9 #ponto auxiliar para secante

res_i = []

#newton
r_newton = newton_poly(coefs_p3, x0_i, eps_p3)
res_i.append(['Newton', r_newton['Iter'], r_newton.get('Raiz'), r_newton.get('Erro'), r_newton.get('Tempo (s)')])

#secante
r_sec = secante_poly(coefs_p3, x0_i, x1_sec_i, eps_p3)
res_i.append(['Secante', r_sec['Iter'], r_sec.get('Raiz'), r_sec.get('Erro'), r_sec.get('Tempo (s)')])

#MPF
r_mpf = mpf_poly_p3(coefs_p3, x0_i, eps_p3)
res_i.append(['MPF', r_mpf['Iter'], r_mpf.get('Raiz'), r_mpf.get('Erro'), r_mpf.get('Tempo (s)')])

df_i = pd.DataFrame(res_i, columns=['Método', 'Iterações', 'Raiz', 'Erro Final', 'Tempo (s)'])

print("\n" + "="*60)
print(f"CASO (i): x0 = {x0_i} (Longe da raiz/Perto de máx local)")
print("="*60)
display(df_i)


#CASO (ii): x0 = -2.0
x0_ii = -2.0
x1_sec_ii = -2.1 #ponto auxiliar para secante

res_ii = []

#newton
r_newton = newton_poly(coefs_p3, x0_ii, eps_p3)
res_ii.append(['Newton', r_newton['Iter'], r_newton.get('Raiz'), r_newton.get('Erro'), r_newton.get('Tempo (s)')])

#secante
r_sec = secante_poly(coefs_p3, x0_ii, x1_sec_ii, eps_p3)
res_ii.append(['Secante', r_sec['Iter'], r_sec.get('Raiz'), r_sec.get('Erro'), r_sec.get('Tempo (s)')])

#MPF
r_mpf = mpf_poly_p3(coefs_p3, x0_ii, eps_p3)
res_ii.append(['MPF', r_mpf['Iter'], r_mpf.get('Raiz'), r_mpf.get('Erro'), r_mpf.get('Tempo (s)')])

df_ii = pd.DataFrame(res_ii, columns=['Método', 'Iterações', 'Raiz', 'Erro Final', 'Tempo (s)'])

print("\n" + "="*60)
print(f"CASO (ii): x0 = {x0_ii} (Próximo da raiz)")
print("="*60)
display(df_ii)


CASO (i): x0 = -0.8 (Longe da raiz/Perto de máx local)


,Método,Iterações,Raiz,Erro Final,Tempo (s)
0,Newton,17,-2.10380340,0.00000000,0.00004251
1,Secante,11,-2.10380340,0.00000003,0.00002609
2,MPF,8,3392663122795.80126953,39050105767240514273333078799276834816.00000000,0.00003343



CASO (ii): x0 = -2.0 (Próximo da raiz)


,Método,Iterações,Raiz,Erro Final,Tempo (s)
0,Newton,3,-2.10380340,0.00000001,0.00001129
1,Secante,3,-2.10380340,0.00000000,0.00000785
2,MPF,10,90536909045232640.00000000,74212487783180404191209664659495349578160424982...,0.00001731


### Explicação do Exemplo 2

Neste exemplo, observamos como a posição inicial afeta os métodos adaptados para polinômios:

1.  **Função MPF Adaptada (`mpf_poly_p3`)**:
    * Para isolar o $x$ no termo linear, fizemos: $3x = x^3 + 3 \rightarrow x = (x^3 + 3) / 3$.
    * No código, usamos `horner_simples` com coeficientes `[1, 0, 0, 3]` para calcular o numerador, e depois dividimos por 3.
    * **Observação:** É esperado que este MPF **divirja** ou convirja muito lentamente para a raiz negativa (aprox -2.1), pois a derivada da função de iteração $\varphi'(x) = x^2$ é muito maior que 1 perto da raiz (critério de convergência $|\varphi'(x)| < 1$ não é satisfeito).

2.  **Caso (i) $x_0 = -0.8$**:
    * Este ponto está numa região "ruim" para Newton (entre o máximo e mínimo local). O método pode oscilar ou demorar a encontrar o caminho para a raiz negativa.
    * A Secante também pode sofrer se os dois pontos iniciais caírem em declives opostos ou muito suaves.

3.  **Caso (ii) $x_0 = -2.0$**:
    * Este ponto está muito próximo da raiz real ($\approx -2.103$).
    * Aqui, espera-se que **Newton** e **Secante** convirjam em pouquíssimas iterações.